In [1]:
import pytz
import pandas as pd
import requests
from datetime import datetime

In [2]:
incident_id = 'DP2025225957'

In [3]:
url = 'https://services1.arcgis.com/zdB7qR0BtYrg0Xpl/arcgis/rest/services/ODC_CRIME_TRAFFICACCIDENTS5YR_P/FeatureServer/325/query'

In [4]:
params = {
    'where': f"incident_id = '{incident_id}'",
    'outFields': '*',
    # 'resultRecordCount': self.page_size,
    # 'resultOffset': offset,
    'orderByFields': 'reported_date ASC',
    'f': 'json',
}

In [5]:
r = requests.get(url, params=params)
r.raise_for_status()
data = r.json()

features = data.get('features', [])
print(f'Result count: {len(features)}')

Result count: 1


In [6]:
df = pd.DataFrame(features[0]['attributes'], index=[0])

In [7]:
df['reported_date']

0    1746392100000
Name: reported_date, dtype: int64

In [8]:
pd.to_datetime(df['reported_date'], unit='ms', utc=False)

0   2025-05-04 20:55:00
Name: reported_date, dtype: datetime64[ms]

In [9]:
pd.to_datetime(df['reported_date'], unit='ms', utc=True)

0   2025-05-04 20:55:00+00:00
Name: reported_date, dtype: datetime64[ms, UTC]

In [10]:
pd.to_datetime(df['reported_date'], unit='ms', utc=False).dt.tz_localize('America/Denver')

0   2025-05-04 20:55:00-06:00
Name: reported_date, dtype: datetime64[ms, America/Denver]

In [11]:
pd.to_datetime(df['reported_date'], unit='ms', utc=False).dt.tz_localize('America/Denver').dt.tz_convert('UTC')

0   2025-05-05 02:55:00+00:00
Name: reported_date, dtype: datetime64[ms, UTC]

## CSV

In [12]:
df = pd.read_csv('../data/crash_data_raw_20260421.csv', low_memory=False)

In [13]:
df[df.incident_id == incident_id].squeeze()

object_id                                             378672035
incident_id                                        DP2025225957
offense_id                                    DP202522595754412
offense_code                                               5441
offense_code_extension                                        2
top_traffic_accident_offense     TRAF - ACCIDENT - FATAL       
first_occurrence_date                       5/4/2025 8:01:00 PM
last_occurrence_date                        5/4/2025 8:01:00 PM
reported_date                               5/4/2025 8:55:00 PM
incident_address                         700 BLOCK E COLFAX AVE
geo_x                                                 3146622.0
geo_y                                                 1694800.0
geo_lon                                             -104.978624
geo_lat                                               39.739875
district_id                                              6     
precinct_id                             

## Postgres

In [14]:
import os
os.chdir('..')
from scripts.crash_data_analysis import CrashDataAnalysis
cda = CrashDataAnalysis()

In [15]:
# Aggregate fatality crashes by year and day of year

query = f"""
select
reported_date
, reported_date at time zone 'America/Denver' as reported_date_denver
from crashes

where incident_id = '{incident_id}'
"""

one_crash_sql = pd.read_sql(query, cda.conn)

In [16]:
one_crash_sql

,reported_date,reported_date_denver
0,2025-05-05 02:55:00+00:00,2025-05-04 20:55:00
